# Reproduce `test_model_Nacl.ipynb` Results

该 notebook 用与 `test_model_Nacl.ipynb` 一致的方式计算：
- `energy RMSE = sqrt(mean(((ref_energy - pred_energy)/atom_num)^2))`
- `force RMSE  = sqrt(mean((ref_force - pred_force)^2))`

并固定为同一 checkpoint 与同一 `NaCl.xyz`，用于在 `fit-4hdnnp-NaCl-test` 目录中复现实验结果。

In [1]:
import os
import sys
from pathlib import Path
TASK_ROOT = Path("/work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG/fit-4hdnnp-NaCl-test").resolve()
os.chdir(TASK_ROOT)
ROOT_DIR = str(TASK_ROOT.parent)
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

In [2]:
import sys
import os
from pathlib import Path
import numpy as np
import torch
from ase.io import read
import cace

ROOT_DIR = "/work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG"
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

os.environ["PYTHONWARNINGS"] = "ignore"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("ROOT_DIR:", ROOT_DIR)
print("device:", device)

ROOT_DIR: /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG
device: cpu


In [4]:
# 与 test_model_Nacl.ipynb 保持一致的默认路径
MODEL_PATH = Path("/work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG/fit-4hdnnp-NaCl/loss_data/Nacl_model.pth")
DATA_PATH = Path("/work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG/fit-4hdnnp-NaCl/NaCl.xyz")

assert MODEL_PATH.exists(), f"checkpoint not found: {MODEL_PATH}"
assert DATA_PATH.exists(), f"data not found: {DATA_PATH}"

# 关键：weights_only=False（与 test_model_Nacl.ipynb 一致）
model = torch.load(str(MODEL_PATH), map_location=device, weights_only=False)


def patch_sog_from_old_checkpoint(m):
    """旧版 pickle 里 SOGPotential 可能缺 Periodic / wl / sl，与当前 cace/modules/ewald.py 不一致。"""
    if m.__class__.__name__ != "SOGPotential":
        return
    if not hasattr(m, "Periodic"):
        # fit-4hdnnp-NaCl/fit-cace-SOG.py 未传 Periodic，默认为 False（非周期实空间 SOG）
        m.Periodic = False
        print("Patched: SOGPotential.Periodic=False")
    dev = None
    for p in m.parameters():
        dev = p.device
        break
    if dev is None and hasattr(m, "amplitude_1"):
        dev = m.amplitude_1.device
    if dev is None:
        dev = torch.device(device)
    a = getattr(m, "amplitude_1", None)
    s = getattr(m, "shift_1", None)
    if a is None or s is None:
        raise RuntimeError("SOGPotential 缺少 amplitude_1/shift_1，无法补全 wl/sl")
    a = a.to(dev)
    s = s.to(dev)
    if not hasattr(m, "wl") or m.wl is None:
        if m.Periodic:
            pi = torch.as_tensor(torch.pi, dtype=a.dtype, device=dev)
            m.wl = torch.nn.Parameter((a * (torch.sqrt(pi) ** 3) * (s**3)).to(dev))
        else:
            m.wl = torch.nn.Parameter(a.clone())
        print("Patched: SOGPotential.wl")
    if not hasattr(m, "sl") or m.sl is None:
        if m.Periodic:
            m.sl = torch.nn.Parameter((-torch.log(2.0 / s)).to(dev))
        else:
            m.sl = torch.nn.Parameter(s.clone())
        print("Patched: SOGPotential.sl")


for m in model.modules():
    patch_sog_from_old_checkpoint(m)

# 直接把 nn.Module 传给 EvaluateTask（与 test_model 一致）
evaluator = cace.tasks.EvaluateTask(
    model_path=model,
    device=device,
    energy_key="CACE_energy",
    forces_key="CACE_forces",
)

data = read(str(DATA_PATH), ":")
pre = evaluator(data)

print("num structures:", len(data))
print("pred energy shape:", pre["energy"].shape)
print("pred forces shape:", pre["forces"].shape)

Patched: SOGPotential.Periodic=False
Patched: SOGPotential.wl
Patched: SOGPotential.sl
Using CPU
num structures: 5000
pred energy shape: (5000,)
pred forces shape: (82500, 3)


In [6]:
# ASE 3.x 读该 extxyz 时常常不把 energy=/forces 列放进 atoms.info / arrays，因此不能用 a.info["energy"]。
# 下面按 cace/data/extxyz_charge.py 同一规则从文本解析 header 的 energy= 与每原子力（不建邻域，比 read_extxyz_with_charge 快很多）。
z_map = {"Na": 11, "Cl": 17}
atomic_energies = {11: -4417.07609365649, 17: -12516.880649933015}


def load_ref_energy_forces_from_extxyz(path, atomic_energies, z_map):
    energies, atom_nums, forces_blocks = [], [], []
    with open(path, "r") as f:
        while True:
            line = f.readline()
            if not line:
                break
            line = line.strip()
            if not line:
                continue
            nat = int(line)
            header = f.readline().strip()
            energy = None
            if "energy=" in header:
                try:
                    e_str = header.split("energy=")[1].split()[0]
                    energy = float(e_str)
                except Exception:
                    energy = None
            species, fl = [], []
            for _ in range(nat):
                parts = f.readline().split()
                species.append(parts[0])
                vals = [float(x) for x in parts[1:]]
                if len(vals) < 7:
                    raise ValueError("expect pos[3], forces[3], charge[1] per line")
                fl.append(vals[3:6])
            Z = [z_map[s] for s in species]
            if energy is not None and atomic_energies is not None:
                energy -= sum(atomic_energies.get(int(z), 0.0) for z in Z)
            if energy is None:
                raise ValueError("no energy= in frame header")
            energies.append(energy)
            atom_nums.append(nat)
            forces_blocks.append(np.asarray(fl, dtype=np.float64))
    ref_energy = np.asarray(energies, dtype=np.float64)
    atom_num = np.asarray(atom_nums, dtype=np.float64)
    ref_force = np.concatenate(forces_blocks, axis=0)
    return ref_energy, atom_num, ref_force


ref_energy, atom_num, ref_force = load_ref_energy_forces_from_extxyz(
    str(DATA_PATH), atomic_energies, z_map
)

# 与 test_model_Nacl.ipynb 一致公式
energy_rmse = np.sqrt(np.mean(((ref_energy - pre["energy"]) / atom_num) ** 2))
print("Energy RMSE (e/atom):", float(energy_rmse))

Energy RMSE (e/atom): 0.11020747739693827


In [7]:
# ref_force 已在上一格由 extxyz 文本解析得到
force_rmse = np.sqrt(np.mean((ref_force - pre["forces"]) ** 2))
print("Force RMSE:", float(force_rmse))

Force RMSE: 0.17313508932136407
